# D17 — Diagnósticos pré-estimação

## tl;dr

**Pergunta desta etapa:** com os grupos congelados em D14–D16, quais comparações são empiricamente sustentáveis e quais ameaças precisam ser conhecidas antes de qualquer estimação causal?

- A amostra congelada é reproduzida: **129 tratados**, **4.963 never-treated**, **5.092 municípios**, **66.196 município-anos** e **66.195 observações elegíveis**.
- O baseline comum de 2007 revela diferenças grandes de escala: os SMDs de unidades locais e emprego ficam próximos de **0,87–0,94**.
- O score logístico é **exclusivamente diagnóstico**: há interseção de suporte, mas unidades nas caudas — inclusive tratados — exigem cautela. `PROPENSITY_SCORE_DIAGNOSTICO != MATCHING`.
- As trajetórias prévias são heterogêneas entre coortes; slopes em nível são particularmente distintas em 2011–2012. A versão `log1p` é usada somente para diagnosticar escala, sem redefinir o outcome causal.
- População, urbanização, PIB, renda e escolaridade **não estão disponíveis** nos dados processados atuais; não são inventados nem baixados.
- Distância sede-a-sede, arranjos populacionais de 2010 e outliers permanecem diagnósticos. Nenhuma unidade é removida.

**Gates conservadores:**

`D17_DIAGNOSTICOS_PRE_ESTIMACAO_CONCLUIDOS = SIM`  
`D17_SUPORTE_EMPIRICO_DOCUMENTADO = SIM`  
`D17_AMOSTRA_PRONTA_TECNICAMENTE_PARA_IMPLEMENTACAO_DO_ESTIMADOR = NAO`  
`D17_ALERTA_IDENTIFICACAO_IMPORTANTE = SIM`  
`DESENHO_CAUSAL_APROVADO = NAO`

## Contexto, método e fronteiras

Esta é uma **auditoria pré-estimação**, não uma estimação de efeito. Não há ATT, Callaway–Sant'Anna, event-study causal, TWFE causal, matching, escolha de amostra por resultado pós-tratamento ou redefinição de tratamento/outcome/coorte.

Referências metodológicas orientadoras: Callaway & Sant'Anna (2021), Goodman-Bacon (2021), Sun & Abraham (2021) e Roth (2022). Em particular, a ausência de rejeição estatística em leads **não comprova tendências paralelas**: testes de pré-tendência podem ter baixo poder, e condicionar decisões ao fato de “passarem” no pré-teste pode distorcer a análise. Por isso, magnitude, direção, incerteza, suporte e plausibilidade substantiva são lidos conjuntamente.

### Decisões D17

1. Baseline comum = **2007**, anterior à primeira coorte (2009), para comparar todas as unidades sem usar informação pós-tratamento.
2. Medidas CEMPRE variantes no tempo só podem entrar como covariáveis quando congeladas em 2007. Em anos contemporâneos/pós-tratamento são outcomes ou possíveis mediadores.
3. UF/macrorregião são estruturais e usadas descritivamente; o modelo de overlap usa macrorregião, não centenas de dummies de UF.
4. O modelo de tratamento é regressão logística L2 simples, apenas para revelar caudas e interseção empírica. Não define desenho, pesos, matching ou exclusões.
5. Arranjos Populacionais de 2010 são diagnóstico datado: potencialmente contemporâneos/pós para coortes 2009–2010 e possivelmente desatualizados para coortes posteriores.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import diagnostica_pre_estimacao as d17
from visualizacao_ipt import aplicar_tema_ipt, salvar_figura_ipt, estilizar_tabela_ipt, CORES_IPT

PATH_AMOSTRA = ROOT / "data" / "processed" / "amostra_causal_cempre_2007_2019.parquet"
PATH_DISTANCIAS = ROOT / "data" / "processed" / "diagnostico_distancias_spillover_fase_ii.parquet"
PATH_ARRANJOS = ROOT / "data" / "processed" / "diagnostico_arranjos_populacionais_fase_ii.parquet"
OUT_DIAG = ROOT / "outputs" / "diagnostics"
OUT_FIG = ROOT / "outputs" / "figures"

COVARIAVEIS_OVERLAP = [
    "qt_unidades_locais",
    "pessoal_ocupado_total",
    "pessoal_ocupado_assalariado",
    "salario_medio_reais_nominal",
]

def executar_git(*args):
    return subprocess.run(["git", *args], cwd=ROOT, check=True, capture_output=True, text=True).stdout.strip()

def mostrar(df, n=20):
    display(estilizar_tabela_ipt(df.head(n)))

def salvar_figura(fig, nome):
    aplicar_tema_ipt(fig)
    caminho = salvar_figura_ipt(fig, OUT_FIG / f"{nome}.png")
    print(f"Figura preservada em: {caminho.relative_to(ROOT)}")
    return fig

## 1. Proveniência Git e auditoria dos inputs

In [2]:
estado_git = {
    "branch": executar_git("branch", "--show-current"),
    "head": executar_git("rev-parse", "HEAD"),
    "origin_main": executar_git("rev-parse", "origin/main"),
    "status_inicial_observado": executar_git("status", "--short", "--", "."),
}
display(pd.DataFrame({"campo": estado_git.keys(), "valor": estado_git.values()}))

,campo,valor
0,branch,main
1,head,c59022fc15e081411ff09bdf8765bc6347d15b00
2,origin_main,c59022fc15e081411ff09bdf8765bc6347d15b00
3,status_inicial_observado,M README.md\n M docs/playbooks/ESTADO_ATUAL.md...


In [3]:
ARTEFATOS_CONGELADOS = [
    PATH_AMOSTRA,
    ROOT / "outputs" / "diagnostics" / "exclusoes_amostra_causal.csv",
    ROOT / "outputs" / "diagnostics" / "resumo_amostra_causal.csv",
    ROOT / "outputs" / "diagnostics" / "distribuicao_coortes.csv",
    ROOT / "outputs" / "diagnostics" / "cobertura_event_time.csv",
    ROOT / "notebooks" / "02_construcao_amostra_causal.ipynb",
    ROOT / "notebooks" / "03_analise_descritiva_amostra_causal.ipynb",
]
hashes_antes = {str(p.relative_to(ROOT)): d17.sha256_arquivo(p) for p in ARTEFATOS_CONGELADOS}

amostra = pd.read_parquet(PATH_AMOSTRA)
distancias = pd.read_parquet(PATH_DISTANCIAS)
arranjos = pd.read_parquet(PATH_ARRANJOS)
auditoria_input = d17.auditar_input_congelado(amostra)
display(pd.DataFrame({"metrica": auditoria_input.keys(), "valor": [str(v) for v in auditoria_input.values()]}))

,metrica,valor
0,n_tratados,129
1,n_controles,4963
2,n_unidades,5092
3,n_linhas,66196
4,n_linhas_elegiveis,66195
5,sobreposicao_tratado_controle,0
6,unidades_painel_desbalanceado,0
7,coortes,"{2009: 21, 2010: 27, 2011: 66, 2012: 13, 2013: 2}"
8,n_missing_outcome,0
9,status_outcome,"['observado', 'zero_real']"


**FATO OBSERVADO.** A chave município-ano é única; o painel-base 2007–2019 é balanceado; não existe sobreposição entre tratados e controles; `5003900` está ausente; o outcome não tem missing nem sigilo; Cabo Frio/2009 é a única observação inelegível.

## 2. Inventário de covariáveis pré-tratamento

In [4]:
inventario = d17.inventariar_covariaveis_disponiveis(amostra)
inventario.to_csv(OUT_DIAG / "covariaveis_pre_tratamento.csv", index=False, encoding="utf-8-sig")
mostrar(inventario, n=60)

variaveis_ausentes = pd.DataFrame(
    {
        "variavel_desejada": [
            "populacao_municipal", "urbanizacao", "PIB/renda", "escolaridade",
            "infraestrutura municipal", "presenca_federal anterior validada fora da janela",
        ],
        "status_D17": "NAO_DISPONIVEL_NOS_DADOS_PROCESSADOS",
        "decisao": "NAO_INVENTAR_NAO_BAIXAR_NAO_USAR",
    }
)
mostrar(variaveis_ausentes)

,variavel,fonte,tipo,anos_disponiveis,missing,missing_pct,uso_potencial,risco_pos_tratamento,decisao_D17
0,municipio_fonte,cadastro territorial,ESTRUTURAL_FIXA,fixa,0,0.000000,identificação e auditoria,BAIXO,NAO_USAR_COMO_PREDITOR
1,status_territorial,calendário territorial IBGE,ESTRUTURAL_FIXA,2007-2019,0,0.000000,qualidade/cobertura,BAIXO,USAR_APENAS_VALIDACAO
2,uf_codigo,cadastro territorial,ESTRUTURAL_FIXA,fixa,0,0.000000,composição geográfica e efeito fixo apenas diagnóstico,BAIXO,USAR_DESCRITIVAMENTE_E_NO_OVERLAP
3,ano_coorte_candidata,cadastro causal/exposição,POS_TRATAMENTO,fixa ou derivada de 2007-2019,64519,97.466614,"definição/auditoria do tratamento, nunca ajuste","DETERMINA tratamento, coorte ou elegibilidade",REJEITAR_COMO_COVARIAVEL
4,ano_evento_institucional,cadastro causal/exposição,POS_TRATAMENTO,fixa ou derivada de 2007-2019,66183,99.980361,"definição/auditoria do tratamento, nunca ajuste","DETERMINA tratamento, coorte ou elegibilidade",REJEITAR_COMO_COVARIAVEL
5,ano_inicio_observado_censo,cadastro causal/exposição,POS_TRATAMENTO,fixa ou derivada de 2007-2019,64519,97.466614,"definição/auditoria do tratamento, nunca ajuste","DETERMINA tratamento, coorte ou elegibilidade",REJEITAR_COMO_COVARIAVEL
6,ano_transicao,cadastro causal/exposição,POS_TRATAMENTO,fixa ou derivada de 2007-2019,66183,99.980361,"definição/auditoria do tratamento, nunca ajuste","DETERMINA tratamento, coorte ou elegibilidade",REJEITAR_COMO_COVARIAVEL
7,candidato_amostra_principal,cadastro causal/exposição,POS_TRATAMENTO,fixa ou derivada de 2007-2019,64519,97.466614,"definição/auditoria do tratamento, nunca ajuste","DETERMINA tratamento, coorte ou elegibilidade",REJEITAR_COMO_COVARIAVEL
8,coorte_g,cadastro causal/exposição,POS_TRATAMENTO,fixa ou derivada de 2007-2019,64519,97.466614,"definição/auditoria do tratamento, nunca ajuste","DETERMINA tratamento, coorte ou elegibilidade",REJEITAR_COMO_COVARIAVEL
9,elegivel_estimacao_principal,cadastro causal/exposição,POS_TRATAMENTO,fixa ou derivada de 2007-2019,0,0.000000,"definição/auditoria do tratamento, nunca ajuste","DETERMINA tratamento, coorte ou elegibilidade",REJEITAR_COMO_COVARIAVEL


,variavel_desejada,status_D17,decisao
0,populacao_municipal,NAO_DISPONIVEL_NOS_DADOS_PROCESSADOS,NAO_INVENTAR_NAO_BAIXAR_NAO_USAR
1,urbanizacao,NAO_DISPONIVEL_NOS_DADOS_PROCESSADOS,NAO_INVENTAR_NAO_BAIXAR_NAO_USAR
2,PIB/renda,NAO_DISPONIVEL_NOS_DADOS_PROCESSADOS,NAO_INVENTAR_NAO_BAIXAR_NAO_USAR
3,escolaridade,NAO_DISPONIVEL_NOS_DADOS_PROCESSADOS,NAO_INVENTAR_NAO_BAIXAR_NAO_USAR
4,infraestrutura municipal,NAO_DISPONIVEL_NOS_DADOS_PROCESSADOS,NAO_INVENTAR_NAO_BAIXAR_NAO_USAR
5,presenca_federal anterior validada fora da janela,NAO_DISPONIVEL_NOS_DADOS_PROCESSADOS,NAO_INVENTAR_NAO_BAIXAR_NAO_USAR


**DECISÃO METODOLÓGICA.** As medidas CEMPRE podem representar porte econômico no baseline, mas viram outcomes/possíveis mediadores quando medidas depois do tratamento. Portanto, somente a fotografia comum de 2007 é usada no diagnóstico de comparabilidade e overlap. A ausência de população impede chamar esse proxy de “porte populacional”.

## 3. Comparabilidade em níveis e escala municipal

In [5]:
baseline = d17.construir_baseline_comum(amostra, COVARIAVEIS_OVERLAP, ano_base=2007)
baseline["porte_emprego_quintil"] = pd.qcut(
    baseline["pessoal_ocupado_assalariado"],
    q=5,
    labels=["Q1_menor", "Q2", "Q3", "Q4", "Q5_maior"],
    duplicates="drop",
)
balanco = d17.balanco_descritivo_baseline(baseline, COVARIAVEIS_OVERLAP)
balanco.to_csv(OUT_DIAG / "balanco_descritivo_pre_tratamento.csv", index=False, encoding="utf-8-sig")
mostrar(balanco)

,variavel,n_tratados,n_controles,media_tratados,mediana_tratados,desvio_padrao_tratados,p10_tratados,p25_tratados,p75_tratados,p90_tratados,media_controles,mediana_controles,desvio_padrao_controles,p10_controles,p25_controles,p75_controles,p90_controles,smd,smd_absoluto
0,qt_unidades_locais,129,4963,2527.224806,1642.000000,3032.050568,340.600000,713.000000,3056.000000,5559.600000,377.555108,135.000000,1163.682056,35.000000,64.000000,313.000000,723.600000,0.936078,0.936078
1,pessoal_ocupado_total,129,4963,18847.062016,10253.000000,24236.608835,1642.400000,4078.000000,22367.000000,48691.000000,2694.264356,784.000000,10080.918983,237.200000,377.500000,1901.500000,4991.800000,0.870244,0.870244
2,pessoal_ocupado_assalariado,129,4963,15744.674419,8182.000000,20194.567004,1269.800000,3316.000000,19086.000000,41552.600000,2246.382430,638.000000,8618.955604,198.000000,309.000000,1547.500000,4073.400000,0.869405,0.869405
3,salario_medio_reais_nominal,129,4963,812.357054,773.690000,209.306191,579.718000,649.500000,910.980000,1117.856000,713.520828,676.470000,242.945848,477.146000,556.035000,817.510000,977.158000,0.435881,0.435881


In [6]:
base_plot = baseline.assign(
    grupo=baseline["papel_causal"].map(
        {d17.PAPEL_TRATADO: "Tratados", d17.PAPEL_CONTROLE: "Never-treated"}
    ),
    log1p_emprego=np.log1p(baseline["pessoal_ocupado_assalariado"]),
)
fig = px.box(
    base_plot,
    x="grupo",
    y="log1p_emprego",
    points=False,
    color="grupo",
    color_discrete_map={"Tratados": CORES_IPT["AZUL_PRINCIPAL"], "Never-treated": CORES_IPT["AZUL_CLARO"]},
    labels={"grupo": "Grupo", "log1p_emprego": "log1p(pessoal ocupado assalariado), 2007"},
    title="Distribuição do proxy de porte econômico no baseline comum (2007)",
)
fig.update_layout(showlegend=False)
salvar_figura(fig, "d17_01_porte_baseline")
fig.show()

Figura preservada em: outputs\figures\interactive\d17_01_porte_baseline.html


In [7]:
fig = px.bar(
    balanco.sort_values("smd_absoluto"),
    x="smd",
    y="variavel",
    orientation="h",
    color_discrete_sequence=[CORES_IPT["AZUL_PRINCIPAL"]],
    labels={"smd": "Standardized mean difference", "variavel": "Variável (baseline 2007)"},
    title="Diferenças padronizadas entre tratados e never-treated",
)
fig.add_vline(x=0, line_color=CORES_IPT["AZUL_ESCURO"], line_width=1)
salvar_figura(fig, "d17_02_smd_baseline")
fig.show()

composicao_porte = (
    baseline.groupby(["papel_causal", "porte_emprego_quintil"], observed=True)
    .size().rename("n").reset_index()
)
composicao_porte["proporcao"] = composicao_porte["n"] / composicao_porte.groupby("papel_causal")["n"].transform("sum")
mostrar(composicao_porte)

Figura preservada em: outputs\figures\interactive\d17_02_smd_baseline.html


,papel_causal,porte_emprego_quintil,n,proporcao
0,CONTROLE_NEVER_TREATED,Q1_menor,1022,0.205924
1,CONTROLE_NEVER_TREATED,Q2,1016,0.204715
2,CONTROLE_NEVER_TREATED,Q3,1016,0.204715
3,CONTROLE_NEVER_TREATED,Q4,1000,0.201491
4,CONTROLE_NEVER_TREATED,Q5_maior,909,0.183155
5,TRATADO_PRINCIPAL,Q1_menor,2,0.015504
6,TRATADO_PRINCIPAL,Q3,1,0.007752
7,TRATADO_PRINCIPAL,Q4,16,0.124031
8,TRATADO_PRINCIPAL,Q5_maior,110,0.852713


**DIAGNÓSTICO.** A distribuição dos tratados está deslocada para municípios economicamente maiores já em 2007. Isso não reprova automaticamente DiD, mas mostra que comparações sem condicionamento dependem de extrapolação substantiva e que o outcome em nível é dominado por diferenças de porte.

## 4. Outcome pré-tratamento por coorte

In [8]:
amostra_log = amostra.copy()
amostra_log["log1p_outcome_diagnostico"] = np.log1p(amostra_log[d17.OUTCOME_PRIMARIO])

resumos_nivel = pd.concat(
    [d17.comparar_coorte_never_pre(amostra, g) for g in d17.COORTES],
    ignore_index=True,
)
resumos_log = pd.concat(
    [d17.comparar_coorte_never_pre(amostra_log, g, "log1p_outcome_diagnostico") for g in d17.COORTES],
    ignore_index=True,
)
resumos_nivel["escala"] = "nivel_outcome_congelado"
resumos_log["escala"] = "log1p_apenas_diagnostico_escala"
mostrar(resumos_nivel, n=20)

,coorte_g,ano,tempo_relativo_pre,grupo,alerta_tamanho_coorte,n_unidades,media,mediana,desvio_padrao,p10,p25,p75,p90,escala
0,2009,2007,-2,CONTROLES_NEVER_TREATED,SEM_ALERTA_TAMANHO,4963,2246.382430,638.000000,8618.955604,198.000000,309.000000,1547.500000,4073.400000,nivel_outcome_congelado
1,2009,2007,-2,TRATADOS_COORTE,SEM_ALERTA_TAMANHO,21,19857.190476,14627.000000,18209.523417,3188.000000,5505.000000,28928.000000,43996.000000,nivel_outcome_congelado
2,2009,2008,-1,CONTROLES_NEVER_TREATED,SEM_ALERTA_TAMANHO,4963,2322.327826,644.000000,8954.100178,196.000000,315.000000,1612.500000,4211.800000,nivel_outcome_congelado
3,2009,2008,-1,TRATADOS_COORTE,SEM_ALERTA_TAMANHO,21,20236.333333,14970.000000,19003.354999,3686.000000,4887.000000,29158.000000,45451.000000,nivel_outcome_congelado
4,2010,2007,-3,CONTROLES_NEVER_TREATED,SEM_ALERTA_TAMANHO,4963,2246.382430,638.000000,8618.955604,198.000000,309.000000,1547.500000,4073.400000,nivel_outcome_congelado
5,2010,2007,-3,TRATADOS_COORTE,SEM_ALERTA_TAMANHO,27,11842.814815,7041.000000,12131.145092,1232.000000,2400.000000,18516.500000,29295.400000,nivel_outcome_congelado
6,2010,2008,-2,CONTROLES_NEVER_TREATED,SEM_ALERTA_TAMANHO,4963,2322.327826,644.000000,8954.100178,196.000000,315.000000,1612.500000,4211.800000,nivel_outcome_congelado
7,2010,2008,-2,TRATADOS_COORTE,SEM_ALERTA_TAMANHO,27,12032.777778,7520.000000,11806.567955,1424.600000,2972.000000,18596.000000,28317.600000,nivel_outcome_congelado
8,2010,2009,-1,CONTROLES_NEVER_TREATED,SEM_ALERTA_TAMANHO,4963,2449.761032,702.000000,9319.859811,223.000000,350.000000,1720.500000,4457.000000,nivel_outcome_congelado
9,2010,2009,-1,TRATADOS_COORTE,SEM_ALERTA_TAMANHO,26,11884.730769,6683.000000,12565.012145,1308.000000,2500.750000,17402.750000,26929.000000,nivel_outcome_congelado


In [9]:
plot_pre = resumos_nivel.assign(
    grupo_rotulo=resumos_nivel["grupo"].map(
        {"TRATADOS_COORTE": "Tratados da coorte", "CONTROLES_NEVER_TREATED": "Never-treated"}
    )
)
fig = px.line(
    plot_pre,
    x="tempo_relativo_pre",
    y="media",
    color="grupo_rotulo",
    facet_col="coorte_g",
    facet_col_wrap=2,
    markers=True,
    color_discrete_map={"Tratados da coorte": CORES_IPT["AZUL_PRINCIPAL"], "Never-treated": CORES_IPT["AZUL_CLARO"]},
    labels={"tempo_relativo_pre": "k (somente pré)", "media": "Média do outcome", "grupo_rotulo": "Grupo"},
    title="Outcome pré-tratamento em nível por coorte (descritivo)",
    height=850,
)
salvar_figura(fig, "d17_03_outcome_pre_nivel_coortes")
fig.show()

plot_log = resumos_log.assign(
    grupo_rotulo=resumos_log["grupo"].map(
        {"TRATADOS_COORTE": "Tratados da coorte", "CONTROLES_NEVER_TREATED": "Never-treated"}
    )
)
fig = px.line(
    plot_log,
    x="tempo_relativo_pre",
    y="media",
    color="grupo_rotulo",
    facet_col="coorte_g",
    facet_col_wrap=2,
    markers=True,
    color_discrete_map={"Tratados da coorte": CORES_IPT["AZUL_PRINCIPAL"], "Never-treated": CORES_IPT["AZUL_CLARO"]},
    labels={"tempo_relativo_pre": "k (somente pré)", "media": "Média de log1p(outcome)", "grupo_rotulo": "Grupo"},
    title="Outcome pré-tratamento em log1p por coorte — somente diagnóstico de escala",
    height=850,
)
salvar_figura(fig, "d17_04_outcome_pre_log_coortes")
fig.show()

Figura preservada em: outputs\figures\interactive\d17_03_outcome_pre_nivel_coortes.html


Figura preservada em: outputs\figures\interactive\d17_04_outcome_pre_log_coortes.html


## 5. Tendências pré-tratamento: diferenças, slopes e incerteza

In [10]:
diferencas_nivel = pd.concat(
    [d17.diferencas_medias_pre_com_ic(amostra, g) for g in d17.COORTES],
    ignore_index=True,
)
diferencas_log = pd.concat(
    [d17.diferencas_medias_pre_com_ic(amostra_log, g, "log1p_outcome_diagnostico") for g in d17.COORTES],
    ignore_index=True,
)
slopes_nivel = pd.concat(
    [d17.diagnostico_slopes_pre(amostra, g) for g in d17.COORTES],
    ignore_index=True,
)
slopes_log = pd.concat(
    [d17.diagnostico_slopes_pre(amostra_log, g, "log1p_outcome_diagnostico") for g in d17.COORTES],
    ignore_index=True,
)

pre_export = pd.concat(
    [
        diferencas_nivel.assign(tipo_diagnostico="diferenca_media", escala="nivel"),
        diferencas_log.assign(tipo_diagnostico="diferenca_media", escala="log1p"),
        slopes_nivel.assign(tipo_diagnostico="slope_medio", escala="nivel"),
        slopes_log.assign(tipo_diagnostico="slope_medio", escala="log1p"),
    ],
    ignore_index=True,
    sort=False,
)
pre_export.to_csv(OUT_DIAG / "diagnostico_pre_tendencias.csv", index=False, encoding="utf-8-sig")
mostrar(slopes_nivel, n=20)

,coorte_g,grupo,n_unidades_com_slope,slope_medio_pre,erro_padrao,ic95_inferior,ic95_superior,rotulo_inferencia
0,2009,CONTROLES_NEVER_TREATED,4963,75.945396,15.952861,44.677789,107.213003,DIAGNOSTICO_DE_PODER_LIMITADO
1,2009,TRATADOS_COORTE,21,379.142857,309.479258,-227.436488,985.722202,DIAGNOSTICO_DE_PODER_LIMITADO
2,2010,CONTROLES_NEVER_TREATED,4963,101.689301,9.715482,82.646956,120.731646,DIAGNOSTICO_DE_PODER_LIMITADO
3,2010,TRATADOS_COORTE,27,209.240741,267.851128,-315.747470,734.228952,DIAGNOSTICO_DE_PODER_LIMITADO
4,2011,CONTROLES_NEVER_TREATED,4963,131.506810,9.878547,112.144858,150.868762,DIAGNOSTICO_DE_PODER_LIMITADO
5,2011,TRATADOS_COORTE,66,1009.696970,171.680932,673.202343,1346.191596,DIAGNOSTICO_DE_PODER_LIMITADO
6,2012,CONTROLES_NEVER_TREATED,4963,135.564920,8.173595,119.544674,151.585167,DIAGNOSTICO_DE_PODER_LIMITADO
7,2012,TRATADOS_COORTE,13,1637.430769,610.556003,440.741004,2834.120534,DIAGNOSTICO_DE_PODER_LIMITADO
8,2013,CONTROLES_NEVER_TREATED,4963,122.459054,7.433889,107.888631,137.029477,DIAGNOSTICO_DE_PODER_LIMITADO
9,2013,TRATADOS_COORTE,2,2103.771429,1280.885714,-406.764571,4614.307429,DIAGNOSTICO_DE_PODER_LIMITADO


In [11]:
fig = px.scatter(
    diferencas_nivel,
    x="tempo_relativo_pre",
    y="diferenca_media",
    color="coorte_g",
    error_y=diferencas_nivel["ic95_superior"] - diferencas_nivel["diferenca_media"],
    error_y_minus=diferencas_nivel["diferenca_media"] - diferencas_nivel["ic95_inferior"],
    color_continuous_scale=[CORES_IPT["AZUL_CLARO"], CORES_IPT["AZUL_PRINCIPAL"], CORES_IPT["AZUL_ESCURO"]],
    labels={"tempo_relativo_pre": "k (somente pré)", "diferenca_media": "Tratados — controles", "coorte_g": "Coorte"},
    title="Diferenças médias prévias por coorte — diagnóstico não causal",
)
fig.add_hline(y=0, line_dash="dash", line_color=CORES_IPT["AZUL_ESCURO"])
salvar_figura(fig, "d17_05_diferencas_pre_ic")
fig.show()

Figura preservada em: outputs\figures\interactive\d17_05_diferencas_pre_ic.html


**INTERPRETAÇÃO CAUTELOSA.** Slopes e intervalos acima são diagnósticos de poder limitado, sensíveis à escala e à composição. Eles não formam um teste de aprovação/reprovação. A coorte 2009 tem apenas dois períodos pré; 2012 tem 13 tratados e 2013 apenas 2, produzindo grande incerteza. A evidência visual e numérica, sobretudo em 2011–2012, justifica `ALERTA_IMPORTANTE`, não uma conclusão causal.

## 6. Overlap / suporte empírico

In [12]:
scores, coeficientes_logit, meta_overlap = d17.ajustar_propensity_diagnostico(
    baseline,
    covariaveis_numericas=COVARIAVEIS_OVERLAP,
    covariaveis_categoricas=["macrorregiao"],
)
scores = scores.merge(
    baseline[["codigo_municipio_ibge", "porte_emprego_quintil", "pessoal_ocupado_assalariado"]],
    on="codigo_municipio_ibge",
    how="left",
    validate="one_to_one",
)
scores.to_csv(OUT_DIAG / "diagnostico_overlap.csv", index=False, encoding="utf-8-sig")

display(pd.DataFrame({"metrica": meta_overlap.keys(), "valor": [str(v) for v in meta_overlap.values()]}))
mostrar(coeficientes_logit)

,metrica,valor
0,limite_inferior,0.0008337553758523133
1,limite_superior,0.8486715310750602
2,n_tratados,129
3,n_controles,4963
4,n_tratados_fora,5
5,n_controles_fora,1361
6,modelo,regressao_logistica_L2_diagnostica
7,ano_baseline,2007
8,covariaveis_numericas,"['qt_unidades_locais', 'pessoal_ocupado_total'..."
9,covariaveis_categoricas,['macrorregiao']


,termo,coeficiente_logit
0,macrorregiao_Sudeste,-2.257275
1,log1p_z__qt_unidades_locais,2.080576
2,macrorregiao_Sul,-1.473387
3,macrorregiao_Norte,1.161389
4,log1p_z__salario_medio_reais_nominal,-0.350701
5,log1p_z__pessoal_ocupado_assalariado,0.282817
6,macrorregiao_Nordeste,0.183439
7,log1p_z__pessoal_ocupado_total,-0.078613


In [13]:
fig = px.histogram(
    scores.assign(grupo=np.where(scores["tratado"], "Tratados", "Never-treated")),
    x="propensity_score",
    color="grupo",
    histnorm="probability density",
    barmode="overlay",
    opacity=0.62,
    nbins=50,
    color_discrete_map={"Tratados": CORES_IPT["AZUL_PRINCIPAL"], "Never-treated": CORES_IPT["AZUL_CLARO"]},
    labels={"propensity_score": "Score de tratamento (diagnóstico)", "grupo": "Grupo"},
    title="Distribuição do score diagnóstico de tratamento — não é matching",
)
fig.add_vline(x=meta_overlap["limite_inferior"], line_dash="dash", line_color=CORES_IPT["AZUL_ESCURO"])
fig.add_vline(x=meta_overlap["limite_superior"], line_dash="dash", line_color=CORES_IPT["AZUL_ESCURO"])
salvar_figura(fig, "d17_06_overlap_propensity")
fig.show()

overlap_regiao = d17.resumir_overlap_por_segmento(scores, "macrorregiao")
overlap_porte = d17.resumir_overlap_por_segmento(scores, "porte_emprego_quintil")
overlap_coorte = d17.resumir_overlap_por_segmento(scores.loc[scores["tratado"]], "coorte_g")
mostrar(overlap_regiao)
mostrar(overlap_porte)
mostrar(overlap_coorte)

Figura preservada em: outputs\figures\interactive\d17_06_overlap_propensity.html


,macrorregiao,tratado,n_unidades,score_min,score_p10,score_mediana,score_p90,score_max,n_fora_suporte
0,Centro-Oeste,False,410,0.000111,0.000634,0.005332,0.054682,0.612779,59
1,Centro-Oeste,True,17,0.009565,0.077748,0.200393,0.653078,0.892030,1
2,Nordeste,False,1578,0.000036,0.000488,0.004781,0.044703,0.838638,265
3,Nordeste,True,41,0.005894,0.023963,0.169296,0.526802,0.962359,2
4,Norte,False,385,0.000102,0.000668,0.006130,0.083304,0.482266,47
5,Norte,True,18,0.000834,0.029555,0.283877,0.789331,0.874397,1
6,Sudeste,False,1518,0.000013,0.000120,0.001121,0.031189,0.848672,662
7,Sudeste,True,31,0.003565,0.018031,0.125045,0.404009,0.655846,0
8,Sul,False,1072,0.000036,0.000247,0.002274,0.037521,0.828910,328
9,Sul,True,22,0.057547,0.086609,0.169672,0.585894,0.877900,1


,porte_emprego_quintil,tratado,n_unidades,score_min,score_p10,score_mediana,score_p90,score_max,n_fora_suporte
0,Q1_menor,False,1022,0.000013,0.000091,0.000413,0.002046,0.020378,722
1,Q1_menor,True,2,0.000834,0.001340,0.003364,0.005388,0.005894,0
2,Q2,False,1016,0.000040,0.000218,0.001102,0.006426,0.105262,431
3,Q3,False,1016,0.000070,0.000620,0.003098,0.016090,0.090120,147
4,Q3,True,1,0.029767,0.029767,0.029767,0.029767,0.029767,0
5,Q4,False,1000,0.000036,0.001324,0.007376,0.045037,0.195431,52
6,Q4,True,16,0.003565,0.006677,0.030456,0.116792,0.141295,0
7,Q5_maior,False,909,0.000142,0.005683,0.031026,0.196297,0.848672,9
8,Q5_maior,True,110,0.017677,0.060056,0.207616,0.656494,0.962359,5


,coorte_g,tratado,n_unidades,score_min,score_p10,score_mediana,score_p90,score_max,n_fora_suporte
0,2009.000000,True,21,0.003565,0.046738,0.146504,0.588434,0.791139,0
1,2010.000000,True,27,0.009565,0.021332,0.146924,0.453702,0.874397,1
2,2011.000000,True,66,0.000834,0.022909,0.168071,0.659085,0.907542,3
3,2012.000000,True,13,0.052482,0.116321,0.394923,0.636553,0.962359,1
4,2013.000000,True,2,0.041811,0.047725,0.071380,0.095035,0.100949,0


`PROPENSITY_SCORE_DIAGNOSTICO != MATCHING`  
O intervalo empírico é uma descrição das distribuições ajustadas no próprio conjunto, não uma regra de trimming. Não há pesos, ESS ponderado, pareamento ou seleção automática; logo, não se reporta ESS artificial sem uma política de pesos definida.

## 7. Composição geográfica

In [14]:
geografia = d17.composicao_geografica(baseline)
regiao = (
    baseline.groupby(["papel_causal", "macrorregiao"], as_index=False)
    .agg(n_municipios=("codigo_municipio_ibge", "nunique"))
)
regiao["proporcao"] = regiao["n_municipios"] / regiao.groupby("papel_causal")["n_municipios"].transform("sum")
mostrar(regiao)

fig = px.bar(
    regiao,
    x="macrorregiao",
    y="proporcao",
    color="papel_causal",
    barmode="group",
    color_discrete_map={d17.PAPEL_TRATADO: CORES_IPT["AZUL_PRINCIPAL"], d17.PAPEL_CONTROLE: CORES_IPT["AZUL_CLARO"]},
    labels={"macrorregiao": "Macrorregião", "proporcao": "Proporção do grupo", "papel_causal": "Papel"},
    title="Composição regional de tratados e never-treated",
)
salvar_figura(fig, "d17_07_composicao_regional")
fig.show()

,papel_causal,macrorregiao,n_municipios,proporcao
0,CONTROLE_NEVER_TREATED,Centro-Oeste,410,0.082611
1,CONTROLE_NEVER_TREATED,Nordeste,1578,0.317953
2,CONTROLE_NEVER_TREATED,Norte,385,0.077574
3,CONTROLE_NEVER_TREATED,Sudeste,1518,0.305863
4,CONTROLE_NEVER_TREATED,Sul,1072,0.215998
5,TRATADO_PRINCIPAL,Centro-Oeste,17,0.131783
6,TRATADO_PRINCIPAL,Nordeste,41,0.317829
7,TRATADO_PRINCIPAL,Norte,18,0.139535
8,TRATADO_PRINCIPAL,Sudeste,31,0.240310
9,TRATADO_PRINCIPAL,Sul,22,0.170543


Figura preservada em: outputs\figures\interactive\d17_07_composicao_regional.html


## 8. Spillovers e Arranjos Populacionais

In [15]:
spillover_painel = d17.classificar_spillover_controles(amostra, distancias)
spillover_unidades = spillover_painel.drop_duplicates("codigo_municipio_ibge")
spillover_controles = spillover_unidades.loc[
    spillover_unidades["papel_causal"].eq(d17.PAPEL_CONTROLE)
].copy()
spillover_controles.to_csv(
    OUT_DIAG / "diagnostico_spillover_controles.csv", index=False, encoding="utf-8-sig"
)
resumo_spillover = pd.DataFrame(
    {
        "limiar_km": [25, 50, 100],
        "n_controles_potencialmente_expostos": [
            int(spillover_controles["fl_ate_25_km"].sum()),
            int(spillover_controles["fl_ate_50_km"].sum()),
            int(spillover_controles["fl_ate_100_km"].sum()),
        ],
    }
)
resumo_spillover["n_controles_restantes_cenario"] = len(spillover_controles) - resumo_spillover["n_controles_potencialmente_expostos"]
resumo_spillover["proporcao_potencialmente_exposta"] = resumo_spillover["n_controles_potencialmente_expostos"] / len(spillover_controles)
mostrar(resumo_spillover)

fig = px.bar(
    resumo_spillover,
    x="limiar_km",
    y="n_controles_potencialmente_expostos",
    text="n_controles_potencialmente_expostos",
    color_discrete_sequence=[CORES_IPT["AZUL_PRINCIPAL"]],
    labels={"limiar_km": "Limiar sede-a-sede (km)", "n_controles_potencialmente_expostos": "Controles sinalizados"},
    title="Controles dentro de limiares diagnósticos de proximidade",
)
salvar_figura(fig, "d17_08_spillover_limiares")
fig.show()

,limiar_km,n_controles_potencialmente_expostos,n_controles_restantes_cenario,proporcao_potencialmente_exposta
0,25,374,4589,0.075358
1,50,1475,3488,0.297199
2,100,3640,1323,0.733427


Figura preservada em: outputs\figures\interactive\d17_08_spillover_limiares.html


In [16]:
arranjos_amostra = (
    baseline[["codigo_municipio_ibge", "papel_causal", "coorte_g", "uf_codigo"]]
    .merge(
        arranjos,
        on="codigo_municipio_ibge",
        how="left",
        validate="one_to_one",
        suffixes=("", "_arranjo"),
    )
)
controles_arranjo = arranjos_amostra.loc[arranjos_amostra["papel_causal"].eq(d17.PAPEL_CONTROLE)]
resumo_arranjo = pd.DataFrame(
    {
        "metrica": [
            "controles_no_mesmo_arranjo_de_fase_ii",
            "controles_fora_dessa_flag",
            "controles_com_arranjo_identificado",
        ],
        "valor": [
            int(controles_arranjo["fl_mesmo_arranjo_populacional_fase_ii"].fillna(False).sum()),
            int((~controles_arranjo["fl_mesmo_arranjo_populacional_fase_ii"].fillna(False)).sum()),
            int(controles_arranjo["fl_pertence_arranjo_populacional"].fillna(False).sum()),
        ],
    }
)
mostrar(resumo_arranjo)

,metrica,valor
0,controles_no_mesmo_arranjo_de_fase_ii,145
1,controles_fora_dessa_flag,-5108
2,controles_com_arranjo_identificado,722


**DIAGNÓSTICO, não exclusão.** Distâncias medem sedes municipais, não campi, fluxos ou tempo de viagem. Arranjos de 2010 não são uma medida pré-tratamento perfeita para todas as coortes. `papel_causal` permanece inalterado; as classes são temporárias e não entram no parquet D15.

**LIMITAÇÃO DE COBERTURA.** O parquet de arranjos disponível tem granularidade de municípios candidatos a controle (4.964 linhas, incluindo o código de sigilo depois excluído da amostra), não dos 129 tratados. Assim, ele identifica 145 controles em arranjos que contêm algum município Fase II, mas não permite tabular de forma defensável “tratados no mesmo arranjo” nem decompor essa exposição por coorte. A D17 registra a não identificabilidade em vez de imputar coortes ou fabricar linhas de tratados.

## 9. Coortes pequenas e Cabo Frio

In [17]:
cobertura_cabo = d17.diagnosticar_cobertura_cabo_frio(amostra)
mostrar(cobertura_cabo)

coortes_pequenas = pd.DataFrame(
    {
        "coorte_g": [2012, 2013],
        "n_tratados": [13, 2],
        "periodos_pre_disponiveis": [5, 6],
        "alerta": ["PRECISAO_LIMITADA", "PRECISAO_MUITO_LIMITADA"],
        "decisao_D17": ["NAO_EXCLUIR", "NAO_EXCLUIR"],
    }
).merge(
    overlap_coorte[["coorte_g", "n_unidades", "n_fora_suporte"]],
    on="coorte_g",
    how="left",
)
mostrar(coortes_pequenas)

,janela,k_min,k_max,n_periodos,n_populacao_calendario,n_janela_completa_elegivel,perda_por_cobertura,perda_cabo_frio,unica_mascara_observacional
0,principal_k_-2_2,-2,2,5,129,128,1,1,CABO_FRIO_2009
1,sensibilidade_k_-3_2,-3,2,6,108,107,1,1,CABO_FRIO_2009


,coorte_g,n_tratados,periodos_pre_disponiveis,alerta,decisao_D17,n_unidades,n_fora_suporte
0,2012,13,5,PRECISAO_LIMITADA,NAO_EXCLUIR,13,1
1,2013,2,6,PRECISAO_MUITO_LIMITADA,NAO_EXCLUIR,2,0


**FATO OBSERVADO — Cabo Frio.** O calendário contém 129 tratados na janela principal e 108 na sensibilidade. A única máscara observacional é Cabo Frio/2009 (`k=-1` para `g=2010`), reduzindo a cobertura elegível completa para 128 e 107, respectivamente. A regra é auditada, não modificada.

## 10. Influência e outliers

In [18]:
extremos, concentracao = d17.diagnostico_influencia_baseline(baseline)
mostrar(concentracao)
mostrar(
    extremos[[
        "codigo_municipio_ibge", "municipio_fonte", "papel_causal",
        "pessoal_ocupado_assalariado", "percentil_no_grupo", "participacao_no_total_grupo",
    ]],
    n=30,
)

top_plot = baseline.sort_values("pessoal_ocupado_assalariado", ascending=False).head(15).sort_values("pessoal_ocupado_assalariado")
fig = px.bar(
    top_plot,
    x="pessoal_ocupado_assalariado",
    y="municipio_fonte",
    color="papel_causal",
    orientation="h",
    color_discrete_map={d17.PAPEL_TRATADO: CORES_IPT["AZUL_PRINCIPAL"], d17.PAPEL_CONTROLE: CORES_IPT["AZUL_CLARO"]},
    labels={"pessoal_ocupado_assalariado": "Pessoal ocupado assalariado (2007)", "municipio_fonte": "Município", "papel_causal": "Papel"},
    title="Maiores municípios no outcome do baseline — influência potencial",
)
salvar_figura(fig, "d17_09_outliers_baseline")
fig.show()

,papel_causal,top_n,participacao_top_n,n_unidades_grupo
0,CONTROLE_NEVER_TREATED,1,0.022636,4963
1,CONTROLE_NEVER_TREATED,5,0.084448,4963
2,CONTROLE_NEVER_TREATED,10,0.133484,4963
3,TRATADO_PRINCIPAL,1,0.070978,129
4,TRATADO_PRINCIPAL,5,0.208573,129
5,TRATADO_PRINCIPAL,10,0.343486,129


,codigo_municipio_ibge,municipio_fonte,papel_causal,pessoal_ocupado_assalariado,percentil_no_grupo,participacao_no_total_grupo
3478,3548708,São Bernardo do Campo - SP,CONTROLE_NEVER_TREATED,252366.000000,1.000000,0.022636
3015,3505708,Barueri - SP,CONTROLE_NEVER_TREATED,214122.000000,0.999799,0.019206
3468,3547809,Santo André - SP,CONTROLE_NEVER_TREATED,169417.000000,0.999597,0.015196
3422,3543402,Ribeirão Preto - SP,CONTROLE_NEVER_TREATED,159861.000000,0.999396,0.014339
3476,3548500,Santos - SP,CONTROLE_NEVER_TREATED,145728.000000,0.999194,0.013071
4294,4305108,Caxias do Sul - RS,TRATADO_PRINCIPAL,144161.000000,1.000000,0.070978
3323,3534401,Osasco - SP,CONTROLE_NEVER_TREATED,134962.000000,0.998993,0.012106
3760,4115200,Maringá - PR,CONTROLE_NEVER_TREATED,107309.000000,0.998791,0.009625
3096,3513801,Diadema - SP,CONTROLE_NEVER_TREATED,104711.000000,0.998590,0.009392
3479,3548807,São Caetano do Sul - SP,CONTROLE_NEVER_TREATED,101543.000000,0.998388,0.009108


Figura preservada em: outputs\figures\interactive\d17_09_outliers_baseline.html


## 11. Tabela de riscos para identificação

In [19]:
max_smd = float(balanco["smd_absoluto"].max())
n_tratados_fora = int(meta_overlap["n_tratados_fora"])
slopes_alerta = slopes_nivel.loc[
    slopes_nivel["coorte_g"].isin([2011, 2012])
    & slopes_nivel["grupo"].eq("TRATADOS_COORTE"),
    ["coorte_g", "slope_medio_pre"],
]

riscos = pd.DataFrame(
    [
        ["diferenças de porte", f"SMD máximo em 2007 = {max_smd:.3f}; população municipal ausente", "ALTA", "SIM", "SIM", "covariáveis externas pré-tratamento e especificações condicionais", "não estimar antes de decisão metodológica"],
        ["suporte limitado", f"{n_tratados_fora} tratados fora do intervalo empírico do score diagnóstico", "ALTA", "SIM", "SIM", "overlap por coorte/região/porte e influência de caudas", "não fazer trimming automático"],
        ["tendências prévias", "slopes em nível 2011–2012 diferem materialmente; log1p reduz efeito de escala sem resolver identificação", "ALTA", "SIM", "SIM", "especificação diagnóstica adicional e avaliação substantiva", "manter alerta; p>0,05 não valida"],
        ["composição regional", "distribuições regionais de tratados e controles não coincidem", "MEDIA", "SIM", "SIM", "comparações condicionais por região/UF", "não impor controle geográfico automaticamente"],
        ["spillover", f"{int(resumo_spillover.loc[resumo_spillover.limiar_km.eq(100), 'n_controles_potencialmente_expostos'].iloc[0])} controles até 100 km; {int(resumo_arranjo.loc[resumo_arranjo.metrica.eq('controles_no_mesmo_arranjo_de_fase_ii'), 'valor'].iloc[0])} no mesmo arranjo", "MEDIA", "SIM", "SIM", "sensibilidades 25/50/100 km e arranjos", "não excluir na D17"],
        ["coortes pequenas", "2012=13 e 2013=2 tratados", "ALTA", "NAO_DIRETAMENTE", "SIM", "inferência/ponderação por coorte", "não excluir; antecipar baixa precisão"],
        ["outliers/influência", "top 10 tratados concentram {:.1%} do outcome tratado em 2007".format(float(concentracao.loc[(concentracao.papel_causal.eq(d17.PAPEL_TRATADO)) & (concentracao.top_n.eq(10)), 'participacao_top_n'].iloc[0])), "MEDIA", "PODE", "SIM", "sensibilidades de influência pré-especificadas", "não excluir automaticamente"],
        ["antecipação", "timing institucional individual é desconhecido para 128/129; suposição principal continua 0", "ALTA", "SIM", "SIM", "evidência institucional e leads", "manter como hipótese, não fato"],
        ["timing anual aproximado", "proxy anual do Censo; Cabo Frio é a única exceção institucional validada", "MEDIA", "SIM", "SIM", "validação institucional adicional", "preservar regra congelada"],
    ],
    columns=["risco", "evidencia_D17", "gravidade", "afeta_identificacao", "afeta_precisao", "diagnostico_futuro", "acao_recomendada"],
)
riscos.to_csv(OUT_DIAG / "riscos_identificacao_D17.csv", index=False, encoding="utf-8-sig")
mostrar(riscos, n=20)

,risco,evidencia_D17,gravidade,afeta_identificacao,afeta_precisao,diagnostico_futuro,acao_recomendada
0,diferenças de porte,SMD máximo em 2007 = 0.936; população municipal ausente,ALTA,SIM,SIM,covariáveis externas pré-tratamento e especificações condicionais,não estimar antes de decisão metodológica
1,suporte limitado,5 tratados fora do intervalo empírico do score diagnóstico,ALTA,SIM,SIM,overlap por coorte/região/porte e influência de caudas,não fazer trimming automático
2,tendências prévias,slopes em nível 2011–2012 diferem materialmente; log1p reduz efeito de escala sem resolver identificação,ALTA,SIM,SIM,especificação diagnóstica adicional e avaliação substantiva,"manter alerta; p>0,05 não valida"
3,composição regional,distribuições regionais de tratados e controles não coincidem,MEDIA,SIM,SIM,comparações condicionais por região/UF,não impor controle geográfico automaticamente
4,spillover,3640 controles até 100 km; 145 no mesmo arranjo,MEDIA,SIM,SIM,sensibilidades 25/50/100 km e arranjos,não excluir na D17
5,coortes pequenas,2012=13 e 2013=2 tratados,ALTA,NAO_DIRETAMENTE,SIM,inferência/ponderação por coorte,não excluir; antecipar baixa precisão
6,outliers/influência,top 10 tratados concentram 34.3% do outcome tratado em 2007,MEDIA,PODE,SIM,sensibilidades de influência pré-especificadas,não excluir automaticamente
7,antecipação,timing institucional individual é desconhecido para 128/129; suposição principal continua 0,ALTA,SIM,SIM,evidência institucional e leads,"manter como hipótese, não fato"
8,timing anual aproximado,proxy anual do Censo; Cabo Frio é a única exceção institucional validada,MEDIA,SIM,SIM,validação institucional adicional,preservar regra congelada


## 12. Matriz de decisão pré-estimação

In [20]:
matriz_decisao = pd.DataFrame(
    [
        ["FATO_OBSERVADO", "Amostra D15/D16 reproduzida sem divergência", "SEM_ALERTA_RELEVANTE"],
        ["FATO_OBSERVADO", "Cabo Frio/2009 é a única máscara; 129/128 e 108/107", "ALERTA_MONITORAR"],
        ["DIAGNOSTICO", "Diferenças grandes de escala no baseline 2007", "ALERTA_IMPORTANTE"],
        ["DIAGNOSTICO", "Overlap existe, mas há tratados e controles em caudas", "ALERTA_IMPORTANTE"],
        ["DIAGNOSTICO", "Pré-trajetórias heterogêneas, especialmente 2011–2012 em nível", "ALERTA_IMPORTANTE"],
        ["AMEACA_PLAUSIVEL", "Spillover geográfico/funcional", "ALERTA_MONITORAR"],
        ["AMEACA_PLAUSIVEL", "Antecipação e timing anual aproximado", "NAO_AVALIAVEL_COM_DADOS_ATUAIS"],
        ["DECISAO_CONGELADA", "Outcome em nível; log1p apenas sensibilidade/escala", "SEM_ALERTA_RELEVANTE"],
        ["DECISAO_CONGELADA", "Never-treated como comparação principal", "SEM_ALERTA_RELEVANTE"],
        ["DECISAO_PENDENTE", "Covariáveis externas, regra de spillover e política para caudas", "ALERTA_IMPORTANTE"],
    ],
    columns=["natureza", "item", "classificacao_D17"],
)
mostrar(matriz_decisao, n=20)

,natureza,item,classificacao_D17
0,FATO_OBSERVADO,Amostra D15/D16 reproduzida sem divergência,SEM_ALERTA_RELEVANTE
1,FATO_OBSERVADO,Cabo Frio/2009 é a única máscara; 129/128 e 108/107,ALERTA_MONITORAR
2,DIAGNOSTICO,Diferenças grandes de escala no baseline 2007,ALERTA_IMPORTANTE
3,DIAGNOSTICO,"Overlap existe, mas há tratados e controles em caudas",ALERTA_IMPORTANTE
4,DIAGNOSTICO,"Pré-trajetórias heterogêneas, especialmente 2011–2012 em nível",ALERTA_IMPORTANTE
5,AMEACA_PLAUSIVEL,Spillover geográfico/funcional,ALERTA_MONITORAR
6,AMEACA_PLAUSIVEL,Antecipação e timing anual aproximado,NAO_AVALIAVEL_COM_DADOS_ATUAIS
7,DECISAO_CONGELADA,Outcome em nível; log1p apenas sensibilidade/escala,SEM_ALERTA_RELEVANTE
8,DECISAO_CONGELADA,Never-treated como comparação principal,SEM_ALERTA_RELEVANTE
9,DECISAO_PENDENTE,"Covariáveis externas, regra de spillover e política para caudas",ALERTA_IMPORTANTE


## 13. Preservação dos artefatos e gates D17

In [21]:
hashes_depois = {str(p.relative_to(ROOT)): d17.sha256_arquivo(p) for p in ARTEFATOS_CONGELADOS}
preservacao = pd.DataFrame(
    {
        "artefato": hashes_antes.keys(),
        "sha256_antes": hashes_antes.values(),
        "sha256_depois": [hashes_depois[k] for k in hashes_antes],
    }
)
preservacao["inalterado"] = preservacao["sha256_antes"].eq(preservacao["sha256_depois"])
mostrar(preservacao, n=20)
assert preservacao["inalterado"].all()

GATES_D17 = {
    "D17_DIAGNOSTICOS_PRE_ESTIMACAO_CONCLUIDOS": "SIM",
    "D17_SUPORTE_EMPIRICO_DOCUMENTADO": "SIM",
    "D17_AMOSTRA_PRONTA_TECNICAMENTE_PARA_IMPLEMENTACAO_DO_ESTIMADOR": "NAO",
    "D17_ALERTA_IDENTIFICACAO_IMPORTANTE": "SIM",
    "DESENHO_CAUSAL_APROVADO": "NAO",
}
display(Markdown("\n".join(f"`{k} = {v}`  " for k, v in GATES_D17.items())))

,artefato,sha256_antes,sha256_depois,inalterado
0,data\processed\amostra_causal_cempre_2007_2019.parquet,7c24c01b569ed6d10f773d00599b5010e1607696f46d716827182a078e4d733b,7c24c01b569ed6d10f773d00599b5010e1607696f46d716827182a078e4d733b,True
1,outputs\diagnostics\exclusoes_amostra_causal.csv,1bdeb015992451732f1b1ffdec0bc6c3bee2bb0f2213aa2421cbeb2659f9347d,1bdeb015992451732f1b1ffdec0bc6c3bee2bb0f2213aa2421cbeb2659f9347d,True
2,outputs\diagnostics\resumo_amostra_causal.csv,a4c9dc7b85b7c17ccd98067ea8eabc2e688582a3fe5fe9894821102860bd1ee1,a4c9dc7b85b7c17ccd98067ea8eabc2e688582a3fe5fe9894821102860bd1ee1,True
3,outputs\diagnostics\distribuicao_coortes.csv,8ecb6b125bd6a3cf02d715748eda52d3518ebcb2fdf33b9af557c4c4db3c59d4,8ecb6b125bd6a3cf02d715748eda52d3518ebcb2fdf33b9af557c4c4db3c59d4,True
4,outputs\diagnostics\cobertura_event_time.csv,c7cfd677ca0818c39d3ca50bc1a4a9e69689cbfa71a0c379633cb0326545947c,c7cfd677ca0818c39d3ca50bc1a4a9e69689cbfa71a0c379633cb0326545947c,True
5,notebooks\02_construcao_amostra_causal.ipynb,b42ac6b4b277bd5b62a266c716b33b7e0dd793ee3827cf2ca3fa11a5f97c4ba0,b42ac6b4b277bd5b62a266c716b33b7e0dd793ee3827cf2ca3fa11a5f97c4ba0,True
6,notebooks\03_analise_descritiva_amostra_causal.ipynb,63825189cb43aa4cbf545ef36f526f06539d500976d4cb06d25d0d67a410629a,63825189cb43aa4cbf545ef36f526f06539d500976d4cb06d25d0d67a410629a,True


`D17_DIAGNOSTICOS_PRE_ESTIMACAO_CONCLUIDOS = SIM`  
`D17_SUPORTE_EMPIRICO_DOCUMENTADO = SIM`  
`D17_AMOSTRA_PRONTA_TECNICAMENTE_PARA_IMPLEMENTACAO_DO_ESTIMADOR = NAO`  
`D17_ALERTA_IDENTIFICACAO_IMPORTANTE = SIM`  
`DESENHO_CAUSAL_APROVADO = NAO`  

## Takeaways

### A. Fatos observados

A população e as regras congeladas D14–D16 são reproduzíveis e foram preservadas por hash. Os dados disponíveis sustentam uma auditoria rica de outcome, geografia, distância e integração funcional, mas não contém população/urbanização/PIB/renda/escolaridade.

### B. Diagnósticos

Tratados são muito maiores economicamente no baseline; o suporte empírico não é completo; a composição regional difere; as pré-trajetórias variam por coorte; os maiores tratados concentram parcela relevante do outcome; e há muitos controles próximos de municípios Fase II sob limiares exploratórios.

### C. Interpretação

Esses sinais tornam arriscado avançar diretamente para uma implementação de ATT como se o desenho estivesse resolvido. Eles não provam violação de tendências paralelas, spillover ou antecipação; tampouco autorizam exclusões automáticas.

### D. Decisões deliberadamente não tomadas

Nenhum efeito causal foi estimado; nenhuma coorte, unidade ou outlier foi excluído; nenhum raio foi escolhido; nenhum matching/trimming/peso foi aplicado; o outcome primário, tratamento, timing, coortes e papéis causais não foram alterados; `DESENHO_CAUSAL_APROVADO` permanece `NAO`.